# 📘 NOTEBOOK 04: DEMAND SENSING & MAPPING (FINAL)
**Mục tiêu:**
1. **Mapping:** Gán ghép dữ liệu bán hàng từ 13 Store gốc (Châu Âu) sang 27 địa điểm thực tế (TP.HCM).
2. **Scaling (Chia nhỏ):** Xử lý bài toán quy mô. Cửa hàng to giữ nguyên, cửa hàng nhỏ (Circle K) chia nhỏ volume.
3. **Depot Assignment (Sửa đổi):** Phân vùng giao hàng chỉ dựa trên **2 Kho (Depot 1 & 3)**, loại bỏ kho Dĩ An theo chiến lược mới.
4. **Output:** File `demand_forecast_hcm.csv` - Đầu vào cho Decision Engine (Notebook 06).

In [1]:
# 1. Setup & Load Data
import pandas as pd
import numpy as np
from google.colab import drive
import os

drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/datastorm_round_2_green_logistics'
DATA_RAW = os.path.join(BASE_PATH, 'data/01_raw')
DATA_PROCESSED = os.path.join(BASE_PATH, 'data/02_processed')

# Load dữ liệu gốc FMCG
print("⏳ Đang load dữ liệu FMCG gốc...")
df_fmcg = pd.read_csv(os.path.join(DATA_RAW, 'fmcg_sales.csv'))
df_fmcg['date'] = pd.to_datetime(df_fmcg['date'])

# Load file mapping đã tạo (stores_hcm_real.csv)
print("⏳ Đang load bảng Mapping 27 Stores...")
df_map = pd.read_csv(os.path.join(DATA_PROCESSED, 'stores_hcm_real.csv'))

# Load file Depots
print("⏳ Đang load danh sách Kho...")
df_depots = pd.read_csv(os.path.join(DATA_PROCESSED, 'depots.csv'))

print(f"✅ FMCG Data: {df_fmcg.shape}")
print(f"✅ Mapping Data: {df_map.shape}")

Mounted at /content/drive
⏳ Đang load dữ liệu FMCG gốc...
⏳ Đang load bảng Mapping 27 Stores...
⏳ Đang load danh sách Kho...
✅ FMCG Data: (1100000, 33)
✅ Mapping Data: (27, 7)


### PHẦN 1: LOGIC SCALING (CHIA NHỎ VOLUME)

In [2]:
# 2. Xử lý Mapping & Scaling Factor

# Hàm xác định Store ID gốc (Parent ID)
# Ví dụ: 'STORE0013_01' -> Parent là 'STORE0013'
# 'STORE0001' -> Parent là 'STORE0001'
def get_parent_id(mapped_id):
    if "_" in mapped_id:
        return mapped_id.split("_")[0]
    return mapped_id

df_map['parent_store_id'] = df_map['store_id_mapped'].apply(get_parent_id)

# Tính Scaling Factor tự động
# Đếm xem 1 Store mẹ đẻ ra bao nhiêu Store con
child_counts = df_map['parent_store_id'].value_counts().to_dict()

def get_scaling_factor(parent_id):
    n_children = child_counts.get(parent_id, 1)
    return 1.0 / n_children

df_map['scaling_factor'] = df_map['parent_store_id'].apply(get_scaling_factor)

print("--- Kiểm tra Logic Scaling ---")
print("Các Store lớn (Hệ số = 1.0):")
print(df_map[df_map['scaling_factor'] == 1.0][['store_id_mapped', 'type', 'scaling_factor']].head(3))
print("\nCác Store nhỏ (Hệ số < 1.0):")
print(df_map[df_map['scaling_factor'] < 1.0][['store_id_mapped', 'type', 'scaling_factor']].head(3))

--- Kiểm tra Logic Scaling ---
Các Store lớn (Hệ số = 1.0):
  store_id_mapped         type  scaling_factor
0       STORE0001  Hypermarket             1.0
1       STORE0004  Hypermarket             1.0
2       STORE0005  Hypermarket             1.0

Các Store nhỏ (Hệ số < 1.0):
   store_id_mapped         type  scaling_factor
12    STORE0013_01  Convenience        0.066667
13    STORE0013_02  Convenience        0.066667
14    STORE0013_03  Convenience        0.066667


### PHẦN 2: TÍNH TOÁN NHU CẦU (DEMAND CALCULATION)
Công thức: `Demand Con` = `Demand Mẹ` * `Hệ số chia` * `Nhiễu ngẫu nhiên`

In [3]:
# 3. Tính toán Demand hàng ngày

# Quy ước: 1 unit trong data gốc ~ 0.5kg (trung bình FMCG)
WEIGHT_PER_UNIT_TON = 0.0005

# Bước A: Tổng hợp Demand của Store Mẹ theo ngày
daily_demand_parent = df_fmcg.groupby(['date', 'store_id'])['units_sold'].sum().reset_index()
daily_demand_parent['parent_weight_ton'] = daily_demand_parent['units_sold'] * WEIGHT_PER_UNIT_TON
daily_demand_parent = daily_demand_parent[['date', 'store_id', 'parent_weight_ton']]
daily_demand_parent.rename(columns={'store_id': 'parent_store_id'}, inplace=True)

# Bước B: Merge với bảng Map để ra 27 dòng/ngày
df_final_demand = pd.merge(df_map, daily_demand_parent, on='parent_store_id', how='left')

# Bước C: Tính Demand KG cuối cùng
df_final_demand['demand_kg'] = df_final_demand['parent_weight_ton'] * df_final_demand['scaling_factor'] * 1000

# Bước D: Thêm nhiễu (Random Noise) cho nhóm Convenience
# Lý do: Để tránh việc 15 cửa hàng Circle K có biểu đồ demand giống hệt nhau (chỉ khác biên độ)
np.random.seed(42)
mask_conv = df_final_demand['type'] == 'Convenience'
# Tạo nhiễu từ 0.85 đến 1.15 (Dao động +/- 15%)
noise = np.random.uniform(0.85, 1.15, size=mask_conv.sum())
df_final_demand.loc[mask_conv, 'demand_kg'] = df_final_demand.loc[mask_conv, 'demand_kg'] * noise

# Làm tròn 2 số thập phân
df_final_demand['demand_kg'] = df_final_demand['demand_kg'].round(2)

print(f"✅ Đã tính xong Demand. Tổng số dòng: {len(df_final_demand)}")
print("Mẫu dữ liệu:")
display(df_final_demand[['date', 'store_id_mapped', 'type', 'demand_kg']].sample(5))

✅ Đã tính xong Demand. Tổng số dòng: 29565
Mẫu dữ liệu:


,date,store_id_mapped,type,demand_kg
26587,2021-11-04,STORE0013_13,Convenience,50.77
4096,2023-03-23,STORE0006,Hypermarket,2624.00
27305,2023-10-23,STORE0013_13,Convenience,43.47
7143,2022-07-28,STORE0002,Supermarket,1955.50
12743,2022-11-30,STORE0012,E-commerce,2257.00


### PHẦN 3: PHÂN KHO (DEPOT ASSIGNMENT) - **LOGIC MỚI**
Chỉ sử dụng **Depot 1 (Q7)** và **Depot 3 (Tân Bình)**. Bỏ qua Depot 2.

In [4]:
# 4. Gán Depot (Chỉ dùng Depot 1 & 3)

# Lọc danh sách kho hoạt động
active_depots = df_depots[df_depots['depot_id'].isin(['DEPOT_001', 'DEPOT_003'])].copy()
print(f"🎯 Các Depot được sử dụng: {active_depots['depot_id'].tolist()}")

# Hàm tính khoảng cách Haversine
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

# Hàm tìm kho gần nhất trong danh sách active
def find_nearest_active_depot(row):
    min_dist = float('inf')
    nearest_depot = None
    for _, depot in active_depots.iterrows():
        dist = haversine(row['lat'], row['long'], depot['lat'], depot['long'])
        if dist < min_dist:
            min_dist = dist
            nearest_depot = depot['depot_id']
    return nearest_depot

# Thực hiện gán
unique_stores = df_final_demand[['store_id_mapped', 'lat', 'long']].drop_duplicates()
unique_stores['assigned_depot'] = unique_stores.apply(find_nearest_active_depot, axis=1)

# Merge kết quả gán kho vào bảng chính
df_final_demand = pd.merge(df_final_demand, unique_stores[['store_id_mapped', 'assigned_depot']], on='store_id_mapped', how='left')

print("✅ Đã phân vùng xong!")
print("Thống kê số lượng cửa hàng theo từng Kho:")
print(df_final_demand.drop_duplicates('store_id_mapped')['assigned_depot'].value_counts())

🎯 Các Depot được sử dụng: ['DEPOT_001', 'DEPOT_003']
✅ Đã phân vùng xong!
Thống kê số lượng cửa hàng theo từng Kho:
assigned_depot
DEPOT_001    19
DEPOT_003     8
Name: count, dtype: int64


In [5]:
# 5. Lưu file kết quả
output_path = os.path.join(DATA_PROCESSED, 'demand_forecast_hcm.csv')
df_final_demand.to_csv(output_path, index=False)

print(f"💾 Đã lưu file thành công tại: {output_path}")
print("👉 Sẵn sàng cho Notebook 06 - Decision Engine!")

💾 Đã lưu file thành công tại: /content/drive/MyDrive/Colab Notebooks/datastorm_round_2_green_logistics/data/02_processed/demand_forecast_hcm.csv
👉 Sẵn sàng cho Notebook 06 - Decision Engine!
